# Download Census 2020 CBG 65+ percentage

In [ ]:
# (1) Imports, API key, and paths
import os
import time
from pathlib import Path
from typing import Iterable

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PROJECT_ROOT = Path.cwd().resolve().parent
OUT_DIR = PROJECT_ROOT / "data"
OUT_FILE = OUT_DIR / "census2020_cbg.csv"

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY", "")
# Get a free key at https://api.census.gov/data/key_signup.html
CENSUS_BASE = "https://api.census.gov/data/2020/dec"
REQUEST_TIMEOUT = 120
SLEEP_SECONDS = 0.15

OUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

print("OUT_FILE:", OUT_FILE)


In [ ]:
# (2) Build the DHC P12 variable list for population age 65+
AGE_65PLUS_GROUPS = {
    "65 and 66 years",
    "67 to 69 years",
    "70 to 74 years",
    "75 to 79 years",
    "80 to 84 years",
    "85 years and over",
}


def census_get_json(endpoint: str, params: dict | None = None) -> object:
    params = dict(params or {})
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY

    response = session.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


variables_json = census_get_json(f"{CENSUS_BASE}/dhc/groups/P12.json")
variables = variables_json["variables"]

p12_65plus_vars = sorted(
    name
    for name, meta in variables.items()
    if name.startswith("P12_")
    and name.endswith("N")
    and not name.endswith("NA")
    and "SEX BY AGE" in meta.get("concept", "").upper()
    and (
        "!!Male:" in meta.get("label", "")
        or "!!Female:" in meta.get("label", "")
    )
    and any(
        meta.get("label", "").strip().endswith(f"!!{age_group}")
        for age_group in AGE_65PLUS_GROUPS
    )
)

if not p12_65plus_vars:
    raise RuntimeError("No DHC P12 age 65+ variables were found.")

print(f"Selected {len(p12_65plus_vars)} variables:")
print(p12_65plus_vars)


In [ ]:
# (3) Download helpers
STATE_FIPS = {
    "AL": "01",
    "AK": "02",
    "AZ": "04",
    "AR": "05",
    "CA": "06",
    "CO": "08",
    "CT": "09",
    "DE": "10",
    "DC": "11",
    "FL": "12",
    "GA": "13",
    "HI": "15",
    "ID": "16",
    "IL": "17",
    "IN": "18",
    "IA": "19",
    "KS": "20",
    "KY": "21",
    "LA": "22",
    "ME": "23",
    "MD": "24",
    "MA": "25",
    "MI": "26",
    "MN": "27",
    "MS": "28",
    "MO": "29",
    "MT": "30",
    "NE": "31",
    "NV": "32",
    "NH": "33",
    "NJ": "34",
    "NM": "35",
    "NY": "36",
    "NC": "37",
    "ND": "38",
    "OH": "39",
    "OK": "40",
    "OR": "41",
    "PA": "42",
    "RI": "44",
    "SC": "45",
    "SD": "46",
    "TN": "47",
    "TX": "48",
    "UT": "49",
    "VT": "50",
    "VA": "51",
    "WA": "53",
    "WV": "54",
    "WI": "55",
    "WY": "56",
}


def census_table_to_frame(rows: list[list[str]]) -> pd.DataFrame:
    if not rows or len(rows) == 1:
        return pd.DataFrame(columns=rows[0] if rows else [])
    return pd.DataFrame(rows[1:], columns=rows[0])


def geoid_from_parts(df: pd.DataFrame) -> pd.Series:
    return df["state"] + df["county"] + df["tract"] + df["block group"]


def fetch_cbg_table(dataset: str, variables: Iterable[str], state_fips: str) -> pd.DataFrame:
    params = {
        "get": ",".join(["NAME", *variables]),
        "for": "block group:*",
        "in": f"state:{state_fips} county:* tract:*",
    }
    rows = census_get_json(f"{CENSUS_BASE}/{dataset}", params=params)
    return census_table_to_frame(rows)


def fetch_all_states(dataset: str, variables: Iterable[str]) -> pd.DataFrame:
    frames = []
    for state_abbr, state_fips in STATE_FIPS.items():
        try:
            print(f"Downloading {dataset.upper()} block groups for {state_abbr}...")
            frames.append(fetch_cbg_table(dataset, variables, state_fips))
        except Exception as exc:
            print(f"Warning: skipped {state_abbr} because {exc}")
        time.sleep(SLEEP_SECONDS)

    if not frames:
        raise RuntimeError(f"No rows were downloaded from {dataset}.")
    return pd.concat(frames, ignore_index=True)


def get_cbg_65plus_counts() -> pd.DataFrame:
    df = fetch_all_states("dhc", p12_65plus_vars)
    value_cols = list(p12_65plus_vars)
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
    return pd.DataFrame({
        "GEOID": geoid_from_parts(df),
        "pop_65plus": df[value_cols].sum(axis=1, min_count=1),
    })


def get_cbg_total_population() -> pd.DataFrame:
    df = fetch_all_states("pl", ["P1_001N"])
    return pd.DataFrame({
        "GEOID": geoid_from_parts(df),
        "total_pop": pd.to_numeric(df["P1_001N"], errors="coerce"),
    })


In [ ]:
# (4) Run CBG download and save output
cbg_65plus = get_cbg_65plus_counts()
cbg_totals = get_cbg_total_population()

census2020_cbg = (
    cbg_65plus.merge(cbg_totals, on="GEOID", how="left")
    .assign(
        prop_65plus=lambda df: (df["pop_65plus"] / df["total_pop"])
        .where(df["total_pop"] > 0)
        .round(6)
    )
    .loc[:, ["GEOID", "prop_65plus"]]
)

census2020_cbg.to_csv(OUT_FILE, index=False)

print("Written:", OUT_FILE)
print(census2020_cbg.info())
census2020_cbg.head()
